# VE SUNAVAL — Superintendencia Nacional de Valores (Venezuela)

**v_2**: pure `requests + BeautifulSoup`. No Selenium needed.

Each of the 9 list pages renders via a DataTables XHR (`/data_<name>.php`) that returns JSON. Each entity's detail lives at `/ficha-empresas/?iqpc=<id>` and is plain server-rendered HTML with a `<table class="table">`. The README's note that the address comes from a modal turned out to be wrong — it's just a separate page reachable by `GET`.

In [1]:
#------------------------------------------------ Begin_Librairie ----------------------------------------
import os
import re
import time
import datetime
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [2]:
#------------------------------------------------ Begin_fileName ----------------------------------------
regulatorName = 'VE SUNAVAL'
now = datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":", ".")[:-7])
scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
os.chdir(scriptfolder)
processdate = now.strftime('%Y-%m-%d')
print(f"Running {regulatorName} v.2 (API-only)")

Running VE SUNAVAL v.2 (API-only)


In [3]:
#------------------------------------------------ Begin_Variable ----------------------------------------
BASE = 'https://www.sunaval.gob.ve'

# ListCode -> (slug, data-endpoint, English list name)
ENDPOINTS = {
    1: ('asesores-de-inversion-jur',              'data_AsesoInver.php',      'List of Investment Advisors (Legal Persons)'),
    2: ('sociedades-de-corretaje-de-inversion',   'data_corretajesycasas.php','List of Investment Brokerage Companies'),
    3: ('casas-de-bolsas-de-productos-agricolas', 'data_InsuAgri.php',        'List of Agricultural Products Brokerage Houses'),
    4: ('fondos-mutuales',                        'data_fondomutual.php',     'List of Mutual Funds'),
    5: ('sociedades-administradoras',             'data_SocieAdm.php',        'List of Management Companies'),
    6: ('sociedades-calificadoras-de-riesgos',    'data_caliriesgo.php',      'List of Risk Rating Companies'),
    7: ('agente-de-traspasos',                    'data_agtraspaso.php',      'List of Transfer Agents'),
    8: ('sociedad-titularizadora',                'data_soctitu.php',         'List of Securitization Companies'),
    9: ('otro-ente',                              'data_otrosentes.php',      'List of Other Entities'),
}

UA = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36'

sqldict = {k: [] for k in [
    'bvdid','priority','ListLabel','Typology','EntryType','Name',
    'InternalID_1','InternalID_1_type','InternalID_2','InternalID_2_type','InternalID_3','InternalID_3_type',
    'CoType','License_Type','Address_1','Address_2','City','Zip','Cntry','Phone','Fax','Website','Email',
    'RegulationType','RegulationTypeCode','RegulationDate','CancellationDate',
    'RegCtry','RegCode','ListCode','ListLanguage','ListValidityDate','ListName','ListProcessDate',
    'LEI Code','BIC SWIFT Code','Name - Mother Company',
    'Address_1 - Mother company','Address_2 -  Mother company','City - Mother company','Zip - Mother company','Cntry - Mother company','Phone - Mother company',
    'Check']}

In [4]:
#------------------------------------------------ Begin_Helpers ----------------------------------------
IQPC_RE = re.compile(r'iqpc=([A-Za-z0-9+/=_-]+)')

def fetch_list(slug, ep):
    """GET the DataTables JSON for one list page."""
    url = f'{BASE}/{ep}?_={int(time.time()*1000)}'
    headers = {'User-Agent': UA, 'X-Requested-With': 'XMLHttpRequest',
               'Accept': 'application/json, text/javascript, */*; q=0.01',
               'Referer': f'{BASE}/{slug}/'}
    r = requests.get(url, headers=headers, timeout=30)
    r.raise_for_status()
    return r.json().get('aaData', [])

def fetch_detail(iqpc):
    """GET the ficha-empresas detail page and parse its label/value table.
    Returns a dict keyed by lower-cased Spanish label."""
    if not iqpc:
        return {}
    url = f'{BASE}/ficha-empresas/?iqpc={iqpc}'
    r = requests.get(url, headers={'User-Agent': UA}, timeout=30)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, 'html.parser')
    table = soup.find('table', class_='table')
    fields = {}
    if table:
        for tr in table.find_all('tr'):
            th, td = tr.find('th'), tr.find('td')
            if th and td:
                fields[th.get_text(strip=True).lower()] = td.get_text(' ', strip=True)
    return fields

def pad(d):
    """Pad every list in d to the length of ListProcessDate."""
    n = len(d['ListProcessDate'])
    for k in d:
        if len(d[k]) < n:
            d[k] += [''] * (n - len(d[k]))
    return d

In [5]:
#------------------------------------------------ Begin_Main ----------------------------------------
for code, (slug, ep, list_name) in ENDPOINTS.items():
    print(f'[INFO] L{code} {slug}: fetching listing...')
    entities = fetch_list(slug, ep)
    print(f'  -> {len(entities)} entities')

    for i, row in enumerate(entities, 1):
        m = IQPC_RE.search(row.get('codigo', '') or '')
        iqpc = m.group(1) if m else None
        detail = fetch_detail(iqpc)

        sqldict['Name'].append(detail.get('nombre') or row.get('nombre', ''))
        sqldict['InternalID_1'].append(detail.get('r.i.f') or row.get('rif', ''))
        sqldict['InternalID_1_type'].append('RIF')
        sqldict['InternalID_2'].append(detail.get('código alfa', ''))
        sqldict['InternalID_2_type'].append('Codigo Alfa' if detail.get('código alfa') else '')
        sqldict['InternalID_3'].append(detail.get('nro. expediente', ''))
        sqldict['InternalID_3_type'].append('Nro. Expediente' if detail.get('nro. expediente') else '')
        sqldict['Address_1'].append(detail.get('dirección', ''))
        sqldict['RegulationType'].append(detail.get('estatus') or row.get('estado') or 'Regulated')
        sqldict['RegulationTypeCode'].append(detail.get('nro. resolución de inscripción', ''))
        sqldict['RegulationDate'].append(detail.get('fecha de inscripción', ''))
        sqldict['Typology'].append(' - '.join(filter(None, [detail.get('sector', ''), detail.get('sub-sector', '')])))
        sqldict['ListProcessDate'].append(processdate)
        sqldict['ListName'].append(list_name)
        sqldict['RegCtry'].append('VE')
        sqldict['RegCode'].append('SUNAVAL')
        sqldict['ListCode'].append(str(code))
        sqldict['Cntry'].append('VE')

        sqldict = pad(sqldict)
        time.sleep(0.3)  # polite pacing

print(f'\n[INFO] Total entities collected: {len(sqldict["Name"])}')

[INFO] L1 asesores-de-inversion-jur: fetching listing...
  -> 8 entities
[INFO] L2 sociedades-de-corretaje-de-inversion: fetching listing...
  -> 37 entities
[INFO] L3 casas-de-bolsas-de-productos-agricolas: fetching listing...
  -> 13 entities
[INFO] L4 fondos-mutuales: fetching listing...
  -> 7 entities
[INFO] L5 sociedades-administradoras: fetching listing...
  -> 5 entities
[INFO] L6 sociedades-calificadoras-de-riesgos: fetching listing...
  -> 9 entities
[INFO] L7 agente-de-traspasos: fetching listing...
  -> 6 entities
[INFO] L8 sociedad-titularizadora: fetching listing...
  -> 6 entities
[INFO] L9 otro-ente: fetching listing...
  -> 5 entities

[INFO] Total entities collected: 96


In [6]:
#------------------------------------------------ Save DataFrame to Excel ----------------------------------------
df = pd.DataFrame(sqldict)
df['RegulationType'] = df['RegulationType'].replace({'Activo': 'Regulated'})

df.to_excel(filename, 'SQL Ready', index=False)
# print(f"[INFO] Excel saved: '{filename}' ({len(df)} rows)")
# print(df.groupby('ListCode').size().to_string())

C:\Users\wuj1\AppData\Local\Temp\2\ipykernel_2140\3317074625.py:5: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(filename, 'SQL Ready', index=False)
